# Introduction

---

This notebook runs LLMs with Bedrock provider which supports tool usage. This is used with langchain with structured output.

## Imports

In [ ]:
import sys

sys.path.append("..")

In [ ]:
import os
import yaml
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
from itertools import repeat
import json
from tqdm import tqdm
from typing import Dict, Union, List, Optional

In [ ]:
os.environ["LANGCHAIN_TRACING_V2"] = "false"
os.environ["LANGSMITH_TRACING"] = "false"
os.environ.pop("LANGCHAIN_API_KEY", None)
os.environ.pop("LANGSMITH_API_KEY", None)

In [ ]:
from src.data_loader.data_loader import get_flatten_radqa_data

from src.utils import get_message_for_llm
from src.few_shot import sample_demonstrations, render_demonstrations

from src.providers.llm_response import get_response_from_llm
from src.providers.llm_langchain_providers import (
    get_bedrock_llm_with_structured_op,
)
from src.logging import RadQALLMEvalLogger

In [ ]:
from src.prompts.bedrock.deepseek_prompt import DEEPSEEK_PROMPT
from src.prompts.bedrock.claude_prompt import CLAUDE_PROMPT
from src.prompts.bedrock.llama_prompt import LLAMA_PROMPT
from src.prompts.bedrock.gpt_oss_prompt import GPT_OSS_PROMPT
from src.prompts.bedrock.mistral_prompt import MISTRAL_PROMPT

## Data Loading

In [ ]:
DATA_PATH = os.path.abspath("../data/radqa/1.0.0/")
assert os.path.exists(DATA_PATH)

In [ ]:
TRAIN_FILE_PATH = os.path.join(DATA_PATH, "train.json")
DEV_FILE_PATH = os.path.join(DATA_PATH, "dev.json")
TEST_FILE_PATH = os.path.join(DATA_PATH, "test.json")

assert os.path.exists(TRAIN_FILE_PATH)
assert os.path.exists(DEV_FILE_PATH)
assert os.path.exists(TEST_FILE_PATH)

In [ ]:
K_SHOT = 1
DEMO_SEED = 20260719

results_subdir = f"results-{K_SHOT}shot"
RESULTS_DIR = os.path.abspath(f"../{results_subdir}/")
os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
LOG_DIR_PATH = os.path.abspath("../logs/")
assert os.path.exists(LOG_DIR_PATH)

In [ ]:
with open("../config.yaml", "r") as fp:
    config_data = yaml.safe_load(fp)

## Logger Setup

In [ ]:
LOG_FILE_PATH = os.path.join(LOG_DIR_PATH, config_data["app"]["log_file_name"])

In [ ]:
logger_instance = RadQALLMEvalLogger(
    app_name=config_data["app"]["app_name"], log_file_path=LOG_FILE_PATH
)
app_logger = logger_instance.get_logger()

In [ ]:
app_logger.info(f"Config Data: {config_data}")

## Process LLM

In [ ]:
BEDROCK_PROMPT_MAPPER = {
    "claude": CLAUDE_PROMPT,
    "llama": LLAMA_PROMPT,
    "mistral": MISTRAL_PROMPT,
    "deepseek": DEEPSEEK_PROMPT,
    "gpt_oss": GPT_OSS_PROMPT,
}

In [ ]:
def find_rate_limiter(runnable):
    """Walk a LangChain Runnable to find an attached rate_limiter."""
    if hasattr(runnable, "rate_limiter"):
        return runnable.rate_limiter
    if hasattr(runnable, "bound"):
        return find_rate_limiter(runnable.bound)
    if hasattr(runnable, "steps"):
        for step in runnable.steps:
            found = find_rate_limiter(step)
            if found is not None:
                return found
    return None

In [ ]:
def wrapper(
    data_subset_file_path: str,
    bedrock_models: List[Dict[str, Union[str, Dict[str, Union[str, int]]]]],
    result_path: str,
    start_idx: int = 0,
    end_idx: Optional[int] = None,
    demonstrations_block: str = "",
) -> None:
    data_subset_name = os.path.splitext(os.path.basename(data_subset_file_path))[0]
    print(f"Data Subset Name: {data_subset_name}")
    radqa_data_flatten_df = get_flatten_radqa_data(file_path=data_subset_file_path)
    print(radqa_data_flatten_df.shape)
    if end_idx is None:
        end_idx = radqa_data_flatten_df.shape[0]
    radqa_data_flatten_records = radqa_data_flatten_df.to_dict(orient="records")[
        start_idx:end_idx
    ]
    credentials_profile_name = bedrock_models["credentials_profile_name"]
    models = bedrock_models["models"]
    for model_family, model_details in models.items():
        system_prompt = BEDROCK_PROMPT_MAPPER[model_family]
        for model_info in model_details:
            model_id = model_info["model_id"]
            region_name = model_info["region_name"]
            model_config = model_info["model_config"]
            endpoint_url = model_info.get("base_url", None)
            messages_batch = [
                get_message_for_llm(
                    model_id=model_id,
                    context=item["context"],
                    question=item["question"],
                    system_prompt=system_prompt,
                    demonstrations_block=demonstrations_block,
                )
                for item in radqa_data_flatten_records
            ]
            print(f"Processing for Model Id: {model_id}")
            bedrock_llm = get_bedrock_llm_with_structured_op(
                model_id=model_id,
                model_config=model_config,
                credentials_profile_name=credentials_profile_name,
                region_name=region_name,
                endpoint_url=endpoint_url,
            )
            app_logger.info(
                f"Model {model_id} | base_url={endpoint_url} | "
                f"rate_limiter={find_rate_limiter(bedrock_llm)}"
            )
            max_workers = 1 if model_family == "llama" else 4
            with ThreadPoolExecutor(max_workers=max_workers) as executor:
                results = list(
                    tqdm(
                        executor.map(
                            get_response_from_llm,
                            repeat(model_id),
                            messages_batch,
                            repeat(bedrock_llm),
                            repeat(app_logger),
                        ),
                        total=len(messages_batch),
                    )
                )

            merged_records_to_save = [
                {**input, **response}
                for input, response in zip(radqa_data_flatten_records, results)
            ]
            safe_model_id = model_id.replace(":", "_").replace("/", "_")
            results_dir = os.path.join(result_path, safe_model_id)
            os.makedirs(results_dir, exist_ok=True)
            with open(
                f"{results_dir}/{data_subset_name}_{start_idx}_{end_idx}.json", "w"
            ) as fp:
                json.dump(merged_records_to_save, fp, indent=4)

### Train Set

In [ ]:
# wrapper(
#     data_subset_file_path=TRAIN_FILE_PATH,
#     bedrock_models=config_data["bedrock"],
#     result_path=RESULTS_DIR,
# )

### Dev Set

In [ ]:
# wrapper(
#     data_subset_file_path=DEV_FILE_PATH,
#     bedrock_models=config_data["bedrock"],
#     result_path=RESULTS_DIR,
# )

# Sample {k} Shot Examples

In [ ]:
train_df_for_demos = get_flatten_radqa_data(file_path=TRAIN_FILE_PATH)
demos = sample_demonstrations(train_df_for_demos, k=K_SHOT, seed=DEMO_SEED)
demonstrations_block = render_demonstrations(demos)
print(
    f"K_SHOT={K_SHOT}; demonstrations block length (chars): {len(demonstrations_block)}"
)

In [ ]:
demonstrations_block

# Debug

In [ ]:
# sample_records = [
#     {
#         "context": "",
#         "question": "",
#     }
# ]
# BEDROCK_PROMPT_MAPPER = {
#     "claude": CLAUDE_PROMPT,
#     "llama": LLAMA_PROMPT,
#     "mistral": MISTRAL_PROMPT,
#     "deepseek": DEEPSEEK_PROMPT,
#     "gpt_oss": GPT_OSS_PROMPT,
# }
# bedrock_models = config_data["bedrock"]
# credentials_profile_name = bedrock_models["credentials_profile_name"]
# models = bedrock_models["models"]
# for model_family, model_details in models.items():
#     system_prompt = BEDROCK_PROMPT_MAPPER[model_family]
#     for model_info in model_details:
#         model_id = model_info["model_id"]
#         region_name = model_info["region_name"]
#         model_config = model_info["model_config"]
#         endpoint_url = model_info.get("base_url", None)
#         messages_batch = [
#             get_message_for_llm(
#                 model_id=model_id,
#                 context=item["context"],
#                 question=item["question"],
#                 system_prompt=system_prompt,
#                 demonstrations_block=demonstrations_block,
#             )
#             for item in sample_records
#         ]
#         print(messages_batch)
#         print(f"Processing for Model Id: {model_id}")
#         bedrock_llm = get_bedrock_llm_with_structured_op(
#             model_id=model_id,
#             model_config=model_config,
#             credentials_profile_name=credentials_profile_name,
#             region_name=region_name,
#             endpoint_url=endpoint_url,
#         )
#         app_logger.info(
#             f"Model {model_id} | base_url={endpoint_url} | "
#             f"rate_limiter={find_rate_limiter(bedrock_llm)}"
#         )
#         max_workers = 1 if model_family == "llama" else 4
#         with ThreadPoolExecutor(max_workers=max_workers) as executor:
#             results = list(
#                 tqdm(
#                     executor.map(
#                         get_response_from_llm,
#                         repeat(model_id),
#                         messages_batch,
#                         repeat(bedrock_llm),
#                         repeat(app_logger),
#                     ),
#                     total=len(messages_batch),
#                 )
#             )

#         merged_records_to_save = [
#             {**input, **response} for input, response in zip(sample_records, results)
#         ]
#         print(f"=====/n{json.dumps(merged_records_to_save)}")

### Test Set

Few-shot demos are drawn from the train split once (fixed seed) and reused across every model and every test query.

## Test connection

In [ ]:
start_idx = 0
end_idx = 1
wrapper(
    data_subset_file_path=TEST_FILE_PATH,
    bedrock_models=config_data["bedrock"],
    result_path=RESULTS_DIR,
    start_idx=start_idx,
    end_idx=end_idx,
    demonstrations_block=demonstrations_block,
)

### Test Set

Few-shot demos are drawn from the train split once (fixed seed) and reused across every model and every test query.

In [ ]:
# Run in batches.
# start_idx - end_idx
# 0 - 300
# 300 - 600
# 600 - None

In [ ]:
start_idx = 0
end_idx = 300
wrapper(
    data_subset_file_path=TEST_FILE_PATH,
    bedrock_models=config_data["bedrock"],
    result_path=RESULTS_DIR,
    start_idx=start_idx,
    end_idx=end_idx,
    demonstrations_block=demonstrations_block,
)

In [ ]:
start_idx = 300
end_idx = 600
wrapper(
    data_subset_file_path=TEST_FILE_PATH,
    bedrock_models=config_data["bedrock"],
    result_path=RESULTS_DIR,
    start_idx=start_idx,
    end_idx=end_idx,
    demonstrations_block=demonstrations_block,
)

In [ ]:
start_idx = 600
end_idx = None
wrapper(
    data_subset_file_path=TEST_FILE_PATH,
    bedrock_models=config_data["bedrock"],
    result_path=RESULTS_DIR,
    start_idx=start_idx,
    end_idx=end_idx,
    demonstrations_block=demonstrations_block,
)